# Publication figures: group mixture linear

This notebook **reuses the same experiments** as `eval.ipynb` (same forwards, metrics, samplers). The only intent here is **paper order**, **knobs for titles/labels**, **`SAVE_FIGURES`**, and optional **bootstrap bands** — not new science.

Run cells **top to bottom** once per checkpoint (section **1** sets paths/imports/plotting). Knob blocks start with `# ---- knobs`.

**Order:** (1) environment → (2) MSE by position + baselines (interactive metric cell) → (3) unit-vector probe → (4) target-component sweep (predict-all) → (5) padding sweep + optional facet ghosts → (6) attention → (7) head patch → (8) prediction lens.


## 1. Environment

Same role as the **first setup cells** in `eval.ipynb`, kept short: **paths** (`PROJECT_DIR`, `SRC_DIR`), **conda / Python** sanity, **legacy protobuf** workaround and optional pin, optional **pip** fallback, **IPython** `%matplotlib inline` and `%autoreload`, quick **import smoke**, then publication matplotlib/seaborn defaults and **`SAVE_FIGURES`**.

**Paths:** `PROJECT_DIR` defaults to `~/icl-time-series` (override with env `ICL_TS_PROJECT`). **`SRC_DIR`** resolves in order: env `ICL_TS_SRC` if it contains `eval.py`, else `cwd`, else `PROJECT_DIR/src`, else `~/icl-time-series/src`. Checkpoint paths in section 2 still use `~/models/<task>/` unless you change that cell.

**Figures:** Section 1 sets **sans-serif (Arial/Helvetica stack)**, a **light y-grid**, **`pub_color`**, **`pub_context_boundary`**, **`pub_legend`** (legend below or outside axes), and default **`image.cmap`** for heatmaps. Adjust those helpers once to affect every plot.


In [ ]:
# ---- knobs (section 1) ----
SAVE_FIGURES = False  # set True to write PNGs for every figure cell
FIGURE_DIR = "figures_pub"  # relative to cwd, or set an absolute path
FIGURE_DPI = 300

# eval.ipynb-style: default repo root ~/icl-time-series (override with env ICL_TS_PROJECT).
AUTO_PIP_INSTALL_MISSING = False  # last resort; prefer conda / environment.yml
AUTO_PIN_PROTOBUF_LEGACY = False  # True: best-effort pip protobuf<=3.20.3 (old wandb/protobuf stacks)
ENABLE_AUTORELOAD = True  # False for a frozen "Run All" publication pass
ENABLE_MATPLOTLIB_INLINE = True  # %matplotlib inline in notebooks

import os
import sys
import json
import copy
import subprocess

USER_HOME = os.path.expanduser("~")
PROJECT_DIR = os.environ.get("ICL_TS_PROJECT", "").strip() or os.path.join(USER_HOME, "icl-time-series")
ICL_TS_SRC = os.environ.get("ICL_TS_SRC", "").strip()

# Harmless on modern stacks; matches eval.ipynb workaround for older protobuf / wandb.
os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION", "python")

if AUTO_PIN_PROTOBUF_LEGACY:
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "protobuf<=3.20.3", "--quiet", "--user"],
            check=False,
            timeout=180,
        )
    except Exception as _e:
        print("AUTO_PIN_PROTOBUF_LEGACY: skipped ({})".format(_e))


def _resolve_src_dir():
    if ICL_TS_SRC and os.path.isfile(os.path.join(ICL_TS_SRC, "eval.py")):
        return os.path.abspath(ICL_TS_SRC)
    here = os.path.abspath(os.getcwd())
    for p in (here, os.path.join(PROJECT_DIR, "src"), os.path.join(USER_HOME, "icl-time-series", "src")):
        if os.path.isfile(os.path.join(p, "eval.py")):
            return p
    return here


SRC_DIR = _resolve_src_dir()
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

_conda = os.environ.get("CONDA_PREFIX")
print("SRC_DIR:", SRC_DIR)
print("PROJECT_DIR:", PROJECT_DIR)
if _conda:
    print("conda:", _conda)
else:
    print("conda: (none) — activate your env if imports fail (see environment.yml).")
print("python:", sys.executable)

_ip = None
try:
    from IPython import get_ipython

    _ip = get_ipython()
except Exception:
    pass
if _ip is not None:
    if ENABLE_MATPLOTLIB_INLINE:
        _ip.run_line_magic("matplotlib", "inline")
    if ENABLE_AUTORELOAD:
        _ip.run_line_magic("load_ext", "autoreload")
        _ip.run_line_magic("autoreload", "2")
    _ipy_parts = []
    if ENABLE_MATPLOTLIB_INLINE:
        _ipy_parts.append("matplotlib inline")
    if ENABLE_AUTORELOAD:
        _ipy_parts.append("autoreload 2")
    if _ipy_parts:
        print("IPython:", ", ".join(_ipy_parts))
elif ENABLE_AUTORELOAD or ENABLE_MATPLOTLIB_INLINE:
    print("IPython not available; skipped inline/autoreload.")

if AUTO_PIP_INSTALL_MISSING:
    _specs = [
        ("numpy", "numpy"),
        ("torch", "torch"),
        ("pandas", "pandas"),
        ("matplotlib", "matplotlib"),
        ("seaborn", "seaborn"),
        ("tqdm", "tqdm"),
        ("pyyaml", "yaml"),
        ("munch", "munch"),
    ]
    for pip_name, imp in _specs:
        try:
            __import__(imp)
        except ImportError:
            print("pip install:", pip_name)
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", pip_name, "--quiet"],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
            )

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import pandas as pd
from tqdm.auto import tqdm

from eval import get_model_from_run
from samplers import get_data_sampler
from tasks import get_task_sampler
from oracle_utils import compute_all_group_mixture_baselines_mse_by_position

try:
    import eval as _eval_mod  # noqa: F401
    print("OK: eval from", os.path.dirname(_eval_mod.__file__))
except ImportError as e:
    raise ImportError(
        "Could not import project modules. cd to repo src/, or set ICL_TS_SRC to that path, "
        "or activate your conda env (see environment.yml)."
    ) from e

# Compact import check (eval.ipynb printed each module; we only warn on failure).
_missing_core = []
for _mod in ("yaml", "munch"):
    try:
        __import__(_mod)
    except ImportError:
        _missing_core.append(_mod)
if _missing_core:
    print("WARNING: missing modules:", _missing_core, "(use conda / environment.yml or AUTO_PIP_INSTALL_MISSING)")

# Global plot style (used by every figure cell below)
sns.set_theme(
    context="paper",
    style="whitegrid",
    font_scale=1.0,
    rc={
        "axes.grid": True,
        "axes.grid.axis": "y",
        "grid.alpha": 0.28,
        "grid.color": "0.82",
        "axes.edgecolor": "0.28",
        "axes.linewidth": 0.9,
    },
)
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans", "Bitstream Vera Sans", "sans-serif"],
    "mathtext.fontset": "dejavusans",
    "figure.figsize": (6.8, 4.0),
    "figure.dpi": 120,
    "savefig.dpi": FIGURE_DPI,
    "savefig.bbox": "tight",
    "axes.labelsize": 10.5,
    "axes.titlesize": 11,
    "legend.fontsize": 7,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "lines.linewidth": 1.65,
    "lines.markersize": 3.5,
    "image.cmap": "cividis",
})

# Colorblind-friendly palette (softer than default tab)
PUB_LINE_COLORS = [
    "#0072B2", "#E69F00", "#009E73", "#CC79A7", "#56B4E9", "#D55E00", "#8C564B", "#333333",
]


def pub_color(i):
    return PUB_LINE_COLORS[int(i) % len(PUB_LINE_COLORS)]


def pub_context_boundary(ax, x, label=None):
    kw = dict(
        color="#111111",
        linestyle=(0, (5, 3)),
        linewidth=2.0,
        zorder=20,
        alpha=0.95,
    )
    if label:
        kw["label"] = label
    ax.axvline(x, **kw)


def pub_legend(fig, ax, where="below", ncol=None):
    """Compact legend: `below` (wide plots, many series) or `outside_right` (few series)."""
    handles, labels = ax.get_legend_handles_labels()
    if not handles:
        fig.tight_layout()
        return
    fs = float(mpl.rcParams.get("legend.fontsize", 7))
    if where == "below":
        ncol = ncol or min(5, max(2, (len(handles) + 2) // 3))
        ax.legend(
            handles,
            labels,
            loc="upper center",
            bbox_to_anchor=(0.5, -0.2),
            ncol=int(ncol),
            fontsize=fs,
            frameon=True,
            framealpha=0.96,
            edgecolor="0.72",
            handlelength=1.5,
            columnspacing=0.85,
        )
        fig.subplots_adjust(bottom=0.26)
        fig.tight_layout()
    elif where == "outside_right":
        ax.legend(
            handles,
            labels,
            loc="upper left",
            bbox_to_anchor=(1.02, 1.0),
            borderaxespad=0.0,
            fontsize=fs,
            frameon=True,
            framealpha=0.96,
            edgecolor="0.72",
        )
        fig.tight_layout(rect=[0.0, 0.0, 0.76, 1.0])
    else:
        ax.legend(fontsize=fs, loc="best", frameon=True)
        fig.tight_layout()


os.makedirs(FIGURE_DIR, exist_ok=True)
_fig_counter = {"n": 0}


def save_pub_figure(fig, stem: str):
    """Save if SAVE_FIGURES; stem should be filesystem-safe."""
    if not SAVE_FIGURES:
        return
    _fig_counter["n"] += 1
    safe = stem.replace(" ", "_").replace("/", "-")
    path = os.path.join(FIGURE_DIR, f"{_fig_counter['n']:02d}_{safe}.png")
    fig.savefig(path)
    print(f"Saved: {path}")


def bootstrap_mean_ci_clip(curves, n_boot=400, alpha=0.05, clip_low=0.0):
    """
    curves: (n_trials, n_positions) -- e.g. MSE curves per trial.
    Returns mean, lower, upper with lower clipped at clip_low (MSE cannot be negative).
    """
    curves = np.asarray(curves, dtype=float)
    if curves.ndim == 1:
        curves = curves.reshape(1, -1)
    n_t, n_p = curves.shape
    mean = np.nanmean(curves, axis=0)
    if n_t < 3:
        return mean, mean, mean
    rng = np.random.default_rng(0)
    boots = np.empty((n_boot, n_p), dtype=float)
    for b in range(n_boot):
        idx = rng.integers(0, n_t, size=n_t)
        boots[b] = np.nanmean(curves[idx], axis=0)
    lo = np.quantile(boots, alpha / 2, axis=0)
    hi = np.quantile(boots, 1 - alpha / 2, axis=0)
    lo = np.maximum(lo, clip_low)
    hi = np.maximum(hi, clip_low)
    return mean, lo, hi


def task_kwargs_from_conf(conf):
    tk = getattr(conf.training, "task_kwargs", None)
    if tk is None:
        return {}
    return dict(tk) if isinstance(tk, dict) else dict(tk.__dict__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(
    "device:", device,
    "| SAVE_FIGURES:", SAVE_FIGURES,
    "| figures:", os.path.abspath(FIGURE_DIR),
)


## 2. Configuration + MSE by position (transformer vs baselines)

Load **`config.yaml`** from the run directory (same as training). **K, C, T** come from `task_kwargs` and the sampler — no hard-coded task geometry.

Edit **`RUN_ID`** and optional **`CHECKPOINT_STEP`** (`-1` = `state.pt`). Baseline **display names** and **which curves to plot** are lists/dicts in the knob block.

In [ ]:
# ---- knobs (section 2: load + main MSE-by-position figure) ----
# USER_HOME is set in section 1 (same as eval.ipynb: ~/…)
RUN_TASK_SUBDIR = "group_mixture_linear"  # folder under ~/models/
RUN_ID = "0092300d-259b-46b0-bfa7-2117e360f05d"
CHECKPOINT_STEP = -1  # -1: state.pt; else state_STEP.pt / model_STEP.pt

N_TRIALS_MSE = 5
BATCH_SIZE_EVAL = None  # None -> use training batch_size from config
BOOTSTRAP_CI = True
BOOTSTRAP_ALPHA = 0.05
BOOTSTRAP_CI_BASELINES = True  # bootstrap bands per baseline (needs N_TRIALS_MSE >= 3)

# Which oracle/baseline keys to draw (order = legend order). Edit freely.
BASELINE_KEYS_ORDER = [
    "ground_truth",
    "true_w_unknown_assignment_bayesian",
    "bayesian_mixture",
    "pure_ls_target",
    "hybrid_bayesian_ls",
]
# Optional prettier labels (key -> string). Leave empty to use defaults below.
BASELINE_LABEL_OVERRIDES = {
    # "bayesian_mixture": "Your label here",
}
_BASELINE_DEFAULT_LABELS = {
    "ground_truth": "Oracle (known w, known assignment)",
    "true_w_unknown_assignment_bayesian": "Bayes (known w, unknown assignment)",
    "bayesian_mixture": "Mixture Bayes (LS context + Bayes target)",
    "pure_ls_target": "LS target only",
    "hybrid_bayesian_ls": "Hybrid Bayes–LS",
}

FIG_TITLE_MSE = "MSE by position"
FIG_XLABEL_MSE = "Sequence index"
FIG_YLABEL_MSE = "Mean squared error"
TRANSFORMER_LEGEND = "Transformer"

run_dir = os.path.join(USER_HOME, "models", RUN_TASK_SUBDIR)
run_path = os.path.join(run_dir, RUN_ID)
if "REPLACE" in RUN_ID:
    raise ValueError("Set RUN_ID to your trained run UUID folder name.")

model, conf = get_model_from_run(run_path, step=CHECKPOINT_STEP)
model.eval().to(device)

task_kwargs = task_kwargs_from_conf(conf)
n_dims = int(conf.model.n_dims)
bs = int(BATCH_SIZE_EVAL or conf.training.batch_size)

# Main eval: match training layout (predict_target_only from config unless overridden here)
OVERRIDE_PREDICT_TARGET_ONLY = None  # e.g. True; None = use config
if OVERRIDE_PREDICT_TARGET_ONLY is not None:
    task_kwargs = dict(task_kwargs)
    task_kwargs["predict_target_only"] = OVERRIDE_PREDICT_TARGET_ONLY

data_sampler = get_data_sampler(conf.training.data, n_dims=n_dims, **task_kwargs)
task_sampler = get_task_sampler(
    conf.training.task,
    n_dims,
    bs,
    num_tasks=getattr(conf.training, "num_tasks", None),
    **task_kwargs,
)
seq_struct = data_sampler.get_sequence_structure()
predict_inds = seq_struct["predict_inds"]
n_points = seq_struct["total_length"]
K = int(data_sampler.n_components)
C = int(data_sampler.contexts_per_component)
T_tgt = int(data_sampler.target_cluster_context_points)
context_length = K * C

metric = None
trial_curves = []
baseline_trials = {k: [] for k in BASELINE_KEYS_ORDER}

for trial in tqdm(range(N_TRIALS_MSE), desc="MSE trials"):
    seed0 = 50_000 + trial * max(bs, 1)
    xs = data_sampler.sample_xs(
        n_points=n_points,
        b_size=bs,
        seeds=list(range(seed0, seed0 + bs)),
    )
    task = task_sampler(
        components=data_sampler.current_components,
        component_assignments=data_sampler.component_assignments,
    )
    ys = task.evaluate(xs)
    if metric is None:
        metric = task.get_metric()

    # Same forward branch as eval.ipynb (metric / MSE-by-position cell).
    with torch.no_grad():
        if predict_inds is not None and len(predict_inds) > 0 and seq_struct is not None:
            if len(predict_inds) == ys.shape[1] and set(predict_inds) == set(range(ys.shape[1])):
                pred = model(xs.to(device), ys.to(device))
            else:
                pred = model(
                    xs.to(device),
                    ys.to(device),
                    inds=predict_inds,
                    sequence_structure=seq_struct,
                )
        else:
            pred = model(xs.to(device), ys.to(device))

    if predict_inds is not None and len(predict_inds) > 0:
        if len(predict_inds) == ys.shape[1]:
            err = metric(pred, ys.to(pred.device))
        else:
            err = metric(pred, ys.to(pred.device)[:, predict_inds])
    else:
        err = metric(pred, ys.to(pred.device))

    # Same reduction as eval.ipynb: batch-mean MSE vector per trial.
    v = np.atleast_1d(err.detach().float().cpu().numpy().mean(axis=0)).astype(float).ravel()
    trial_curves.append(v)

    scale_v = float(getattr(task, "scale", getattr(data_sampler, "scale", 1.0)))
    noise_v = float(getattr(task, "noise_std", getattr(data_sampler, "noise_std", 0.0)))
    mb = compute_all_group_mixture_baselines_mse_by_position(
        xs.cpu(), ys.cpu(),
        data_sampler.current_components,
        data_sampler.component_assignments,
        K, C, T_tgt, scale_v,
        target_noise_std=noise_v,
    )
    for k in BASELINE_KEYS_ORDER:
        if k not in mb:
            continue
        baseline_trials[k].append(mb[k].astype(float))

trial_curves = np.stack(trial_curves, axis=0)
mean_mse, lo_mse, hi_mse = bootstrap_mean_ci_clip(trial_curves, alpha=BOOTSTRAP_ALPHA)
if not BOOTSTRAP_CI:
    lo_mse = hi_mse = mean_mse

L = int(mean_mse.shape[0])
if L == n_points:
    x_tf = np.arange(n_points, dtype=float)
elif predict_inds is not None and len(predict_inds) == L:
    x_tf = np.asarray(predict_inds, dtype=float)
else:
    x_tf = np.arange(L, dtype=float)

fig, ax = plt.subplots(1, 1, figsize=(8.2, 4.6))
_c_tf = pub_color(0)
if mean_mse.size == 1:
    ax.scatter(
        [float(x_tf[0])],
        [float(mean_mse[0])],
        color=_c_tf,
        s=70,
        zorder=5,
        label=TRANSFORMER_LEGEND,
    )
else:
    _band_ok = np.isfinite(mean_mse) & np.isfinite(lo_mse) & np.isfinite(hi_mse)
    if BOOTSTRAP_CI and _band_ok.any():
        ax.fill_between(
            x_tf, lo_mse, hi_mse, where=_band_ok,
            color=_c_tf, alpha=0.22, label="Transformer CI",
        )
    ax.plot(x_tf, mean_mse, color=_c_tf, marker="o", label=TRANSFORMER_LEGEND)

_boot_bl = BOOTSTRAP_CI_BASELINES and BOOTSTRAP_CI and N_TRIALS_MSE >= 3
_bi = 1
for k in BASELINE_KEYS_ORDER:
    if not baseline_trials[k]:
        continue
    arr = np.stack(baseline_trials[k], axis=0)
    mean_b, lo_b, hi_b = bootstrap_mean_ci_clip(arr, alpha=BOOTSTRAP_ALPHA)
    if not _boot_bl:
        lo_b = hi_b = mean_b
    xb = x_tf if mean_b.shape[0] == x_tf.shape[0] else np.arange(mean_b.shape[0], dtype=float)
    lab = BASELINE_LABEL_OVERRIDES.get(k, _BASELINE_DEFAULT_LABELS.get(k, k))
    c = pub_color(_bi)
    _bi += 1
    _okb = np.isfinite(mean_b) & np.isfinite(lo_b) & np.isfinite(hi_b)
    if _boot_bl and _okb.any():
        ax.fill_between(xb, lo_b, hi_b, where=_okb, color=c, alpha=0.16, label=lab + " CI")
    ax.plot(xb, mean_b, ls="--", color=c, label=lab)

if context_length < n_points:
    pub_context_boundary(ax, context_length - 0.5, label="Context / target boundary")
ax.set_title(FIG_TITLE_MSE)
ax.set_xlabel(FIG_XLABEL_MSE)
ax.set_ylabel(FIG_YLABEL_MSE)
pub_legend(fig, ax, where="below", ncol=5)
save_pub_figure(fig, "mse_by_position_baselines")
plt.show()


print("K, C, T_tgt, n_points:", K, C, T_tgt, n_points)


## 3. Unit-vector probe (target cluster; first point by default)

Coefficient plot only (per-dimension readout on unit vectors vs true **w**). **Knobs** control offset, component, and labels; colors/legend follow section 1.

In [ ]:
# ---- knobs (section 3) ----
PROBE_TARGET_COMPONENT = 0
PROBE_TARGET_CLUSTER_OFFSET = 0  # 0 = first point in target cluster
PROBE_FIXED_CLUSTER = None  # None -> [0,1,...,K-1]
PROBE_SEED = 2027

FIG_TITLE_UNIT = "Unit-vector probe vs true weights"

if conf.training.task != "group_mixture_linear":
    raise RuntimeError("This notebook is for group_mixture_linear.")

torch.manual_seed(PROBE_SEED)
np.random.seed(PROBE_SEED)

probe_kw = dict(task_kwargs)
probe_kw["predict_target_only"] = False
ds_p = get_data_sampler(conf.training.data, n_dims=n_dims, **probe_kw)
ts_p = get_task_sampler(conf.training.task, n_dims, 1, num_tasks=getattr(conf.training, "num_tasks", None), **probe_kw)
ss_p = ds_p.get_sequence_structure()
Ttot = ss_p["total_length"]
Kp = ds_p.n_components
Cp = ds_p.contexts_per_component
ctx0 = Kp * Cp
probe_idx = ctx0 + int(PROBE_TARGET_CLUSTER_OFFSET)
if probe_idx < ctx0 or probe_idx >= Ttot:
    raise ValueError("PROBE_TARGET_CLUSTER_OFFSET out of range")

fixed_ctx = list(range(Kp)) if PROBE_FIXED_CLUSTER is None else PROBE_FIXED_CLUSTER
xs_base = ds_p.sample_xs(
    n_points=Ttot, b_size=1,
    fixed_cluster_assignments=fixed_ctx,
    fixed_target_component=PROBE_TARGET_COMPONENT,
    seeds=[12345],
)
task_p = ts_p(components=ds_p.current_components, component_assignments=ds_p.component_assignments)
scale_p = float(getattr(task_p, "scale", 1.0))
w_all = (ds_p.current_components[0, :, :, 0].cpu() * scale_p).numpy()  # (K, d)
true_w = w_all[PROBE_TARGET_COMPONENT]

md = next(model.parameters()).device
inds_one = [probe_idx]

def fwd_one(xs_t, ys_t):
    with torch.no_grad():
        return model(xs_t.to(md), ys_t.to(md), inds=inds_one, sequence_structure=ss_p)

def eval_x(xv):
    xs_q = xs_base.clone()
    xs_q[0, probe_idx, :] = xv
    ys_q = task_p.evaluate(xs_q)
    pred = fwd_one(xs_q, ys_q)
    return float(pred[0, 0].cpu()), float(ys_q[0, probe_idx].cpu())

unit_pred = []
for j in range(n_dims):
    e = torch.zeros(n_dims)
    e[j] = 1.0
    p, t = eval_x(e)
    unit_pred.append(p)
unit_pred = np.array(unit_pred)

dims = np.arange(n_dims)
fig, ax = plt.subplots(1, 1, figsize=(7.0, 4.2))
ax.plot(dims, unit_pred, "o-", color=pub_color(0), label="pred @ e_j")
ax.plot(dims, true_w, "s--", color=pub_color(1), label=f"true w comp {PROBE_TARGET_COMPONENT}")
for k in range(Kp):
    if k == PROBE_TARGET_COMPONENT:
        continue
    ax.plot(dims, w_all[k], "--", marker="x", lw=1.2, color=pub_color(2 + k), label=f"true w comp {k}")
ax.set_xticks(dims)
ax.set_xticklabels([str(int(j)) for j in dims])
ax.set_xlabel("Dimension")
ax.set_ylabel("Coefficient")
ax.set_title(FIG_TITLE_UNIT)
pub_legend(fig, ax, where="outside_right")
save_pub_figure(fig, "unit_vector_probe")
plt.show()


## 4. MSE by position vs target component (symmetry / appendix)

Same experiment as **`eval.ipynb`** cell *“Test: Context [0,1,...,K-1] → Target component 0 vs 1 vs …”*: `predict_target_only=False`, one **`model(xs, ys)`** forward per batch, `mse_by_pos = (pred-ys).pow(2).mean(dim=0)`.

Optional **`N_TRIALS_TARGET_SWEEP` > 1** adds repeated batches + bootstrap bands (extra vs eval; set to **1** to match a single eval run).


In [ ]:
# ---- knobs (section 4) ----
N_TRIALS_TARGET_SWEEP = 1  # eval.ipynb uses one batch per target; raise for smoother means + CI
TARGET_SWEEP_BS = min(8, bs)
BOOTSTRAP_CI_TARGET_SWEEP = True  # only used when N_TRIALS_TARGET_SWEEP >= 3
FIG_TITLE_TARGET_SWEEP = "MSE by position for each fixed target component"
FIXED_CTX_FOR_SWEEP = None  # None -> [0,...,K-1]

sweep_kw = dict(task_kwargs)
sweep_kw["predict_target_only"] = False
ds_s = get_data_sampler(conf.training.data, n_dims=n_dims, **sweep_kw)
ts_s = get_task_sampler(conf.training.task, n_dims, TARGET_SWEEP_BS, num_tasks=getattr(conf.training, "num_tasks", None), **sweep_kw)
ss_s = ds_s.get_sequence_structure()
T_s = ss_s["total_length"]
K_s = ds_s.n_components
ctx_assign = list(range(K_s)) if FIXED_CTX_FOR_SWEEP is None else FIXED_CTX_FOR_SWEEP

fig, ax = plt.subplots(1, 1, figsize=(8.0, 4.5))
markers = "os^DvP"
x_pos = np.arange(T_s)

for tau in range(K_s):
    curves_tau = []
    for trial in range(N_TRIALS_TARGET_SWEEP):
        s0 = 70_000 + trial * 1000 + tau * 17
        xs = ds_s.sample_xs(
            T_s,
            TARGET_SWEEP_BS,
            fixed_cluster_assignments=ctx_assign,
            fixed_target_component=tau,
            seeds=list(range(s0, s0 + TARGET_SWEEP_BS)),
        )
        task = ts_s(components=ds_s.current_components, component_assignments=ds_s.component_assignments)
        ys = task.evaluate(xs)
        with torch.no_grad():
            pred = model(xs.to(device), ys.to(device))
        sq = (pred - ys.to(pred.device)) ** 2
        mse_pos = sq.mean(dim=0).detach().float().cpu().numpy()
        curves_tau.append(mse_pos)
    curves_tau = np.stack(curves_tau, axis=0)
    if BOOTSTRAP_CI_TARGET_SWEEP and N_TRIALS_TARGET_SWEEP >= 3:
        mean_c, lo_c, hi_c = bootstrap_mean_ci_clip(curves_tau, alpha=BOOTSTRAP_ALPHA)
        _ok = np.isfinite(mean_c) & np.isfinite(lo_c) & np.isfinite(hi_c)
        if _ok.any():
            ax.fill_between(x_pos, lo_c, hi_c, where=_ok, color=pub_color(tau), alpha=0.16)
    else:
        mean_c = np.nanmean(curves_tau, axis=0)
    ax.plot(
        x_pos,
        mean_c,
        marker=markers[tau % len(markers)],
        color=pub_color(tau),
        label=f"Target component {tau}",
    )

pub_context_boundary(ax, K_s * ds_s.contexts_per_component - 0.5, label="Context end")
ax.set_title(FIG_TITLE_TARGET_SWEEP)
ax.set_xlabel(FIG_XLABEL_MSE)
ax.set_ylabel(FIG_YLABEL_MSE)
pub_legend(fig, ax, where="outside_right")
save_pub_figure(fig, "mse_by_position_target_sweep")
plt.show()


## 5. Context padding: MSE by position + optional baselines

For each **i** contexts per cluster (rest zero-padded), target cluster unchanged — same loop as **`eval.ipynb`**.

**Match `eval.ipynb` (defaults):** `PADDING_REUSE_SECTION2_SAMPLERS=True` uses the same `data_sampler`, `task_sampler`, and `seq_struct` as section 2 (like the globals above that cell in `eval.ipynb`). `PADDING_BS=None` uses the same batch size as section 2 (`bs` ≈ your `batch_size`). `PADDING_SAMPLE_SEEDS=None` calls `sample_xs` **without** seeds (random batch). `PADDING_FORWARD_MODE="full"` calls `model(xs_pad, ys_pad)` once per **i** with no `sequence_structure`.

**Different on purpose:** set `PADDING_REUSE_SECTION2_SAMPLERS=False` to rebuild samplers (e.g. `PADDING_OVERRIDE_PREDICT_TARGET_ONLY`). `INCLUDE_BASELINES_PADDING` adds oracle curves. `PADDING_FACET_COLS` controls the small-multiples panel (`0` = skip).


In [ ]:
# ---- knobs (section 5) ----
# To match eval.ipynb padding cell: keep defaults below (reuse section 2, full forward, random batch).
PADDING_REUSE_SECTION2_SAMPLERS = True  # True: use globals data_sampler, task_sampler, seq_struct, bs from section 2
PADDING_BS = None  # None -> same batch size as section 2 (`bs`), like eval.ipynb `batch_size`
PADDING_SAMPLE_SEEDS = None  # None -> random xs (eval.ipynb); else list of length batch

INCLUDE_BASELINES_PADDING = []  # e.g. ["ground_truth", "bayesian_mixture"] or [] to skip
FIG_TITLE_PAD = "MSE by position under context padding"
LINE_LABEL_I = "contexts per cluster = {i}"  # formatted with .format(i=i)

# eval.ipynb: one model(xs_pad, ys_pad) per i, no sequence_structure argument.
PADDING_FORWARD_MODE = "full"  # "full" | "per_position"

# Only used when PADDING_REUSE_SECTION2_SAMPLERS is False (fresh samplers):
PADDING_OVERRIDE_PREDICT_TARGET_ONLY = None  # None | True | False

PADDING_FACET_COLS = 3  # 0 = skip small-multiples figure
FIG_TITLE_PAD_FACET = "Prediction error by position (padding sweep)"

_RUN_PAD = conf.training.task == "group_mixture_linear"
if not _RUN_PAD:
    print("Skipped (task is not group_mixture_linear).")

if _RUN_PAD:
    if PADDING_REUSE_SECTION2_SAMPLERS:
        try:
            ds_f = data_sampler
            ts_f = task_sampler
            ss_f = seq_struct
        except NameError as e:
            raise NameError(
                "PADDING_REUSE_SECTION2_SAMPLERS=True needs section 2 run first "
                "(defines data_sampler, task_sampler, seq_struct, bs)."
            ) from e
        _bs_pad = int(bs if PADDING_BS is None else PADDING_BS)
        if PADDING_OVERRIDE_PREDICT_TARGET_ONLY is not None:
            print("Note: PADDING_OVERRIDE_PREDICT_TARGET_ONLY ignored when reusing section-2 samplers.")
    else:
        pad_kw = dict(task_kwargs)
        if PADDING_OVERRIDE_PREDICT_TARGET_ONLY is not None:
            pad_kw = dict(pad_kw)
            pad_kw["predict_target_only"] = PADDING_OVERRIDE_PREDICT_TARGET_ONLY
        ds_f = get_data_sampler(conf.training.data, n_dims=n_dims, **pad_kw)
        _bs_pad = int(PADDING_BS if PADDING_BS is not None else bs)
        ts_f = get_task_sampler(
            conf.training.task,
            n_dims,
            _bs_pad,
            num_tasks=getattr(conf.training, "num_tasks", None),
            **pad_kw,
        )
        ss_f = ds_f.get_sequence_structure()

    T_f = ss_f["total_length"]
    K_f = int(ds_f.n_components)
    C_f = int(ds_f.contexts_per_component)
    ctx_len_f = K_f * C_f

    if PADDING_SAMPLE_SEEDS is None:
        xs_full = ds_f.sample_xs(T_f, _bs_pad)
    else:
        if len(PADDING_SAMPLE_SEEDS) != _bs_pad:
            raise ValueError(
                "PADDING_SAMPLE_SEEDS must have length equal to batch size "
                "(PADDING_BS or bs when PADDING_BS is None)."
            )
        xs_full = ds_f.sample_xs(T_f, _bs_pad, seeds=list(PADDING_SAMPLE_SEEDS))

    if hasattr(ds_f, "current_components") and ds_f.current_components is not None:
        task0 = ts_f(
            components=ds_f.current_components,
            component_assignments=ds_f.component_assignments,
        )
    else:
        task0 = ts_f()
    ys_full = task0.evaluate(xs_full)

    def mse_curve_padded(i_ctx):
        xs_p = xs_full.clone()
        ys_p = ys_full.clone()
        for k in range(K_f):
            st = k * C_f
            xs_p[:, st + i_ctx : st + C_f, :] = 0.0
            ys_p[:, st + i_ctx : st + C_f] = 0.0
        if PADDING_FORWARD_MODE == "full":
            with torch.no_grad():
                pred = model(xs_p.to(device), ys_p.to(device))
            sq = (pred - ys_p.to(pred.device)) ** 2
            mse_pos = sq.mean(dim=0).detach().float().cpu().numpy()
        elif PADDING_FORWARD_MODE == "per_position":
            mse_pos = np.zeros(T_f, dtype=float)
            for pos in range(T_f):
                with torch.no_grad():
                    pr = model(
                        xs_p.to(device),
                        ys_p.to(device),
                        inds=[pos],
                        sequence_structure=ss_f,
                    )
                tgt = ys_p[:, pos : pos + 1].to(pr.device)
                mse_pos[pos] = ((pr - tgt) ** 2).mean().item()
        else:
            raise ValueError("PADDING_FORWARD_MODE must be 'full' or 'per_position'")
        return mse_pos, xs_p, ys_p

    mse_by_pos_per_i = []
    fig, ax = plt.subplots(1, 1, figsize=(8.2, 4.6))
    positions = np.arange(T_f)
    baseline_pad_curves = {bk: [] for bk in INCLUDE_BASELINES_PADDING}

    for i_ctx in tqdm(range(1, C_f + 1), desc="padding i"):
        mse_pos, xs_p, ys_p = mse_curve_padded(i_ctx)
        mse_by_pos_per_i.append(mse_pos)
        ax.plot(positions, mse_pos, color=pub_color(i_ctx), label=LINE_LABEL_I.format(i=i_ctx))
        scale_v = float(getattr(task0, "scale", getattr(ds_f, "scale", 1.0)))
        noise_v = float(getattr(task0, "noise_std", getattr(ds_f, "noise_std", 0.0)))
        for bk in INCLUDE_BASELINES_PADDING:
            mb = compute_all_group_mixture_baselines_mse_by_position(
                xs_p.cpu(),
                ys_p.cpu(),
                ds_f.current_components,
                ds_f.component_assignments,
                K_f,
                C_f,
                int(ds_f.target_cluster_context_points),
                scale_v,
                target_noise_std=noise_v,
            )
            baseline_pad_curves[bk].append(mb[bk].astype(float))

    if ctx_len_f < T_f:
        pub_context_boundary(ax, ctx_len_f - 0.5, label="Context end")

    for _bk_i, bk in enumerate(INCLUDE_BASELINES_PADDING):
        curves_b = np.stack(baseline_pad_curves[bk], axis=0)  # (n_i, T_f)
        mean_b = curves_b.mean(axis=0)
        lab = BASELINE_LABEL_OVERRIDES.get(bk, _BASELINE_DEFAULT_LABELS.get(bk, bk))
        ax.plot(
            positions,
            mean_b,
            ls="--",
            lw=1.5,
            color=pub_color(C_f + 2 + _bk_i),
            label=f"{lab} (mean over i)",
        )

    ax.set_title(FIG_TITLE_PAD)
    ax.set_xlabel(FIG_XLABEL_MSE)
    ax.set_ylabel(FIG_YLABEL_MSE)
    pub_legend(fig, ax, where="below", ncol=6)
    save_pub_figure(fig, "padding_mse_by_position")
    plt.show()

    if PADDING_FACET_COLS > 0:
        cols = int(PADDING_FACET_COLS)
        rows = int((C_f + cols - 1) // cols)
        fig2, _axes_arr = plt.subplots(rows, cols, figsize=(3.8 * cols, 2.75 * rows), sharex=True, sharey=True)
        axes_flat = np.ravel(_axes_arr).tolist()
        for idx, i_ctx in enumerate(range(1, C_f + 1)):
            axf = axes_flat[idx]
            curve_idx = i_ctx - 1
            for ghost in mse_by_pos_per_i:
                axf.plot(positions, ghost, color="0.55", alpha=0.1, lw=1.0)
            axf.plot(
                positions,
                mse_by_pos_per_i[curve_idx],
                "o-",
                ms=3,
                color=pub_color(i_ctx),
                label=f"i={i_ctx}",
            )
            if ctx_len_f < T_f:
                pub_context_boundary(axf, ctx_len_f - 0.5)
            axf.set_title(f"i = {i_ctx}", fontsize=10)
        for j in range(C_f, len(axes_flat)):
            fig2.delaxes(axes_flat[j])
        fig2.suptitle(FIG_TITLE_PAD_FACET, y=1.01, fontsize=12)
        fig2.tight_layout()
        save_pub_figure(fig2, "padding_mse_facets")
        plt.show()


## 6. Attention maps by layer

**Knobs:** which layer(s), batch index, head average or single head. Extend later (e.g. facet by target component).

In [ ]:
# ---- knobs (section 6) ----
ATTN_LAYERS_TO_PLOT = [0, 2, 5, 8, 11]  # edit; must be < n_layer
ATTN_BATCH_IDX = 0
ATTN_AVERAGE_HEADS = True
ATTN_HEAD_IDX = 0
FIG_TITLE_ATTN = "Attention weights (query vs key)"

ds_a = get_data_sampler(conf.training.data, n_dims=n_dims, **task_kwargs)
ss_a = ds_a.get_sequence_structure()
T_a = ss_a["total_length"]
xs_a = ds_a.sample_xs(T_a, min(bs, 4), seeds=list(range(6000, 6000 + min(bs, 4))))
task_a = task_sampler(components=ds_a.current_components, component_assignments=ds_a.component_assignments)
ys_a = task_a.evaluate(xs_a)

with torch.no_grad():
    attns = model.get_attention(xs_a.to(device), ys_a.to(device), ss_a)
n_layer = len(attns)
layers = [ell for ell in ATTN_LAYERS_TO_PLOT if 0 <= ell < n_layer]
if not layers:
    layers = [min(5, n_layer - 1)]

n_pl = len(layers)
fig, axes = plt.subplots(1, n_pl, figsize=(3.2 * n_pl + 1, 3.8), constrained_layout=True)
if n_pl == 1:
    axes = [axes]
vmax = 0.0
mats = []
for ax, ell in zip(axes, layers):
    a = attns[ell][ATTN_BATCH_IDX]
    if ATTN_AVERAGE_HEADS:
        a = a.mean(dim=0).cpu().numpy()
    else:
        a = a[ATTN_HEAD_IDX].cpu().numpy()
    mats.append(a)
    vmax = max(vmax, float(a.max()))
for ax, a, ell in zip(axes, mats, layers):
    im = ax.imshow(a, cmap="cividis", aspect="auto", vmin=0.0, vmax=max(vmax, 1e-8))
    ax.set_title(f"L{ell}")
    ax.set_xlabel("key")
axes[0].set_ylabel("query")
fig.suptitle(FIG_TITLE_ATTN)
fig.colorbar(im, ax=axes, shrink=0.75, label="weight")
save_pub_figure(fig, "attention_layers")
plt.show()


## 7. Head patch importance (c_proj column ablation)

Same mechanism as exploratory `eval.ipynb`: temporarily zero **c_proj** columns for one head, measure MSE change at query. **Knobs** control layers scanned and batch size.

In [ ]:
# ---- knobs (section 7) ----
PATCH_BS = min(8, bs)
PATCH_LAYERS = None  # None -> all layers
FIG_TITLE_PATCH = "Head importance (Δ MSE under c_proj patch)"

cfg = model._backbone.config
nL, nH, nE = cfg.n_layer, cfg.n_head, cfg.n_embd
assert nE % nH == 0
hd = nE // nH
layers_use = list(range(nL)) if not PATCH_LAYERS else [j for j in PATCH_LAYERS if 0 <= j < nL]

ds_h = get_data_sampler(conf.training.data, n_dims=n_dims, **task_kwargs)
ss_h = ds_h.get_sequence_structure()
Th = ss_h["total_length"]
qh = 2 * (Th - 1)
xs_h = ds_h.sample_xs(Th, PATCH_BS, seeds=list(range(8000, 8000 + PATCH_BS)))
task_h = task_sampler(components=ds_h.current_components, component_assignments=ds_h.component_assignments)
ys_h = task_h.evaluate(xs_h)
yq = ys_h[:, -1].to(device)

def mse_query():
    with torch.no_grad():
        p = model(xs_h.to(device), ys_h.to(device), inds=[Th - 1], sequence_structure=ss_h).squeeze(-1)
    return float(((p - yq) ** 2).mean().item())

imp = np.zeros((len(layers_use), nH), dtype=float)
base_mse = mse_query()
blocks = model._backbone.h
for li, layer_idx in enumerate(tqdm(layers_use, desc="layers")):
    blk = blocks[layer_idx]
    W = blk.attn.c_proj.weight
    W0 = W.data.clone()
    for h in range(nH):
        c0, c1 = h * hd, (h + 1) * hd
        W.data[:, c0:c1] = 0.0
        imp[li, h] = mse_query() - base_mse
        W.data.copy_(W0)

fig, ax = plt.subplots(1, 1, figsize=(6.8 + 0.35 * nH, 4.0))
vmax = float(np.nanmax(np.abs(imp))) + 1e-12
im = ax.imshow(imp, aspect="auto", cmap="coolwarm", vmin=-vmax, vmax=vmax)
ax.set_yticks(range(len(layers_use)))
ax.set_yticklabels([str(layers_use[i]) for i in range(len(layers_use))])
ax.set_xticks(range(nH))
ax.set_xlabel("head")
ax.set_ylabel("layer")
ax.set_title(FIG_TITLE_PATCH + f" (baseline MSE={base_mse:.4f})")
fig.colorbar(im, ax=ax, shrink=0.65, label="Δ MSE")
fig.tight_layout()
save_pub_figure(fig, "head_patch_importance")
plt.show()


## 8. Prediction lens (logit-style readout per layer)

Project residual stream at the **query x-token** through **`_read_out`** (and `ln_f` if present), per layer. **Knobs** control example count and title.

In [ ]:
# ---- knobs (section 8) ----
LENS_N_EX = 8
FIG_TITLE_LENS = "Prediction lens: readout vs layer"

ds_l = get_data_sampler(conf.training.data, n_dims=n_dims, **task_kwargs)
ss_l = ds_l.get_sequence_structure()
Tl = ss_l["total_length"]
qidx = 2 * (Tl - 1)
xs_l = ds_l.sample_xs(Tl, LENS_N_EX, seeds=list(range(9000, 9000 + LENS_N_EX)))
task_l = task_sampler(components=ds_l.current_components, component_assignments=ds_l.component_assignments)
ys_l = task_l.evaluate(xs_l)
true_y = ys_l[:, -1].cpu().numpy()

with torch.no_grad():
    hstates = model.get_residual_stream_by_layer(xs_l.to(device), ys_l.to(device), ss_l)
ln_f = getattr(model._backbone, "ln_f", None)
preds_by_layer = []
for h in hstates:
    hq = h[:, qidx, :]
    if ln_f is not None:
        hq = ln_f(hq)
    out = model._read_out(hq.unsqueeze(1)).squeeze(-1)
    preds_by_layer.append(out.detach().cpu().numpy())
P = np.stack(preds_by_layer, axis=0)  # (L+1, B)
layers_axis = np.arange(P.shape[0])

fig, ax = plt.subplots(1, 1, figsize=(8.0, 4.5))
_n_show = min(6, LENS_N_EX)
for b in range(_n_show):
    c = pub_color(b)
    ax.plot(layers_axis, P[:, b], "o-", ms=3, color=c, label=f"ex {b} readout")
    ax.axhline(true_y[b], color=c, ls=":", lw=1.35, alpha=0.9)
ax.set_xlabel("layer index (0 = embed)")
ax.set_ylabel("y")
ax.set_title(FIG_TITLE_LENS)
pub_legend(fig, ax, where="below", ncol=3)
save_pub_figure(fig, "prediction_lens")
plt.show()
